# Tugas Pengganti UTS  
Rizqy Jauhary Atsaany  
235150300111038  
TKOM - Embedded Artificial Intelligence - B  

# Decision Tree varian CART

In [2]:
import numpy as np
import pandas as pd
from collections import Counter

# Fungsi untuk menghitung Gini Index
def gini_index(y):
    counts = np.bincount(y)
    probabilities = counts / len(y)
    return 1 - np.sum([p**2 for p in probabilities if p > 0])

# Fungsi untuk membagi dataset berdasarkan split
def split_dataset(X, y, feature_index, threshold):
    left_mask = X[:, feature_index] <= threshold
    right_mask = ~left_mask
    return X[left_mask], y[left_mask], X[right_mask], y[right_mask]

# Fungsi untuk menghitung Gini Gain
def gini_gain(X, y, feature_index, threshold):
    parent_gini = gini_index(y)
    X_left, y_left, X_right, y_right = split_dataset(X, y, feature_index, threshold)
    if len(y_left) == 0 or len(y_right) == 0:
        return 0
    child_gini = (len(y_left) / len(y)) * gini_index(y_left) + (len(y_right) / len(y)) * gini_index(y_right)
    return parent_gini - child_gini

# Fungsi untuk mencari splitting terbaik berdasarkan Gini Gain
def best_split(X, y):
    best_gini_gain = 0
    best_feature = None
    best_threshold = None
    n_features = X.shape[1]
    for feature_index in range(n_features):
        thresholds = np.unique(X[:, feature_index])
        for threshold in thresholds:
            gain = gini_gain(X, y, feature_index, threshold)
            if gain > best_gini_gain:
                best_gini_gain = gain
                best_feature = feature_index
                best_threshold = threshold
    return best_feature, best_threshold

# Kelas untuk Decision Tree menggunakan CART
class DecisionTreeCART:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y, depth=0):
        if len(np.unique(y)) == 1:
            return np.unique(y)[0]
        if self.max_depth is not None and depth >= self.max_depth:
            return Counter(y).most_common(1)[0][0]
        feature, threshold = best_split(X, y)
        if feature is None:
            return Counter(y).most_common(1)[0][0]
        X_left, y_left, X_right, y_right = split_dataset(X, y, feature, threshold)
        self.tree = {
            'feature': feature,
            'threshold': threshold,
            'left': self.fit(X_left, y_left, depth + 1),
            'right': self.fit(X_right, y_right, depth + 1)
        }
        return self.tree

    def predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree
        feature = tree['feature']
        threshold = tree['threshold']
        if x[feature] <= threshold:
            return self.predict_one(x, tree['left'])
        else:
            return self.predict_one(x, tree['right'])

    def predict(self, X):
        return np.array([self.predict_one(x, self.tree) for x in X])

# Memuat dataset dari file CSV
from sklearn.model_selection import train_test_split

df = pd.read_csv('data.csv')
X = df.drop(columns=['fail']).values
y = df['fail'].astype(int).values

# Membagi dataset menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Membuat dan melatih model CART
model = DecisionTreeCART()
model.fit(X_train, y_train)

# Memprediksi dataset uji
y_pred = model.predict(X_test)

# Menghitung akurasi
accuracy = np.mean(y_pred == y_test)
print(f'Akurasi Model Decision Tree varian CART: {accuracy * 100:.2f}%')

# Contoh prediksi satu data
sample_data = np.array([X_test[0]])  # Mengambil satu data dari dataset uji
predicted_class = model.predict(sample_data)[0]
print(f'Prediksi untuk sampel: {sample_data[0]} adalah kelas {predicted_class}')


Akurasi Model Decision Tree varian CART: 80.42%
Prediksi untuk sampel: [16  7  5  3  6  5 82  3  6] adalah kelas 1


# Random Forest

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# Load dataset dari file CSV
df = pd.read_csv('data.csv')
X = df.drop(columns=['fail']).values
y = df['fail'].astype(int).values

# Bagi dataset menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Buat model Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Prediksi pada data uji
y_pred = rf.predict(X_test)

# Evaluasi model
accuracy = accuracy_score(y_test, y_pred)
print(f'Akurasi Model Random Forest : {accuracy:.2f}')

sample_data = np.array([X_test[0]])  # Mengambil satu data dari dataset uji
predicted_class = rf.predict(sample_data)[0]
print(f'Prediksi untuk sampel: {sample_data[0]} adalah kelas {predicted_class}')



Akurasi Model Random Forest : 0.88
Prediksi untuk sampel: [16  7  5  3  6  5 82  3  6] adalah kelas 1
